In [ ]:
!pip install anthropic pillow pandas tqdm -q

import os
import json
import base64
import pandas as pd
from PIL import Image
from tqdm import tqdm
import anthropic
from google.colab import drive

print('Done!')

In [ ]:
drive.mount('/content/drive')

IMAGE_FOLDER = '/content/drive/MyDrive/CDT_Dataset/images'
LABEL_FOLDER = '/content/drive/MyDrive/CDT_Dataset/labels'

os.makedirs(LABEL_FOLDER, exist_ok=True)

image_files = sorted([
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff'))
])

print(f'เชื่อม Drive สำเร็จ')
print(f'พบรูปทั้งหมด {len(image_files)} รูป')

In [ ]:
ANTHROPIC_API_KEY = "YOUR_API_KEY_HERE"

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SAVE_PATH = '/content/drive/MyDrive/CDT_Dataset/labels/ccss_labels.json'

if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, 'r') as f:
        labels_data = json.load(f)
    print(f'โหลด progress เดิม: label แล้ว {len(labels_data)} รูป')
else:
    labels_data = {}
    print('เริ่ม label ใหม่')

unlabeled = [f for f in image_files if f not in labels_data]
print(f'เหลือรูปที่ต้อง label: {len(unlabeled)} รูป')
print('Ready!!')

In [ ]:
def encode_image(image_path):
    with Image.open(image_path) as img:
        img.thumbnail((800, 800))
        img = img.convert('RGB')
        img.save('/tmp/temp_img.jpg', 'JPEG', quality=85)
    with open('/tmp/temp_img.jpg', 'rb') as f:
        return base64.standard_b64encode(f.read()).decode('utf-8')

def label_image_with_claude(image_path):
    img_data = encode_image(image_path)

    # แก้ prompt — ไม่ให้มี reasoning เพื่อป้องกัน JSON ขาด
    prompt = """วิเคราะห์รูป Clock Drawing Test ตาม CCSS scoring:

A. เลข 1-12: 2=ครบ/ผิด≤2, 1=ผิด1-2, 0=ผิด3+
B. quadrant แย่สุด: 2=ไม่ผิด, 1=ผิด1-2, 0=ผิด3+
C. ตำแหน่ง+ลำดับ: 2=ถูกทั้งคู่, 1=อยู่วงกลมแต่เรียงผิด, 0=บิดเบี้ยว
D. เข็ม: 2=มีสอง+center, 1=มีสองแต่ผิด, 0=มีแค่อัน/ไม่มี
E. ตำแหน่งเข็ม(11:10): 2=ถูก, 1=ผิดอันเดียว, 0=ผิดทั้งคู่

ตอบ JSON เท่านั้น ห้ามมีข้อความอื่นเด็ดขาด:
{"A":0,"B":0,"C":0,"D":0,"E":0}"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=50,  # แก้จาก 300 → 50 พอแค่ JSON
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": img_data
                    }
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }]
    )

    text = response.content[0].text.strip()
    text = text.replace('```json', '').replace('```', '').strip()

    # แก้ — หา JSON จาก response แม้มีข้อความอื่นปน
    import re
    json_match = re.search(r'\{[^}]+\}', text)
    if json_match:
        text = json_match.group()

    result = json.loads(text)
    return result

In [ ]:
BATCH_SAVE = 10  # บันทึกทุก 10 รูป

print(f'🚀 เริ่ม label {len(unlabeled)} รูป...')
print(f'💰 ค่าใช้จ่ายโดยประมาณ: ${len(unlabeled) * 0.002:.2f}')
print('-' * 50)

errors = []

for i, filename in enumerate(tqdm(unlabeled)):
    try:
        img_path = f'{IMAGE_FOLDER}/{filename}'
        result = label_image_with_claude(img_path)

        # คำนวณคะแนนรวม
        total = result['A'] + result['B'] + result['C'] + result['D'] + result['E']
        result['total'] = total
        result['dementia_risk'] = total < 6

        labels_data[filename] = result

        # บันทึกทุก BATCH_SAVE รูป
        if (i + 1) % BATCH_SAVE == 0:
            with open(SAVE_PATH, 'w') as f:
                json.dump(labels_data, f, ensure_ascii=False, indent=2)
            tqdm.write(f'💾 บันทึก progress: {len(labels_data)} รูป')

    except Exception as e:
        errors.append(filename)
        tqdm.write(f'⚠️ Error: {filename} — {str(e)}')
        continue

# บันทึกครั้งสุดท้าย
with open(SAVE_PATH, 'w') as f:
    json.dump(labels_data, f, ensure_ascii=False, indent=2)

print(f'\nDone!')
print(f'label สำเร็จ: {len(labels_data)} รูป')
print(f'Error: {len(errors)} รูป')
if errors:
    print(f'รูปที่ error: {errors}')